<a href="https://colab.research.google.com/github/b181005/Cellpose-Image-Analysis/blob/main/260323_cellposeAll.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%pip install cellpose matplotlib numpy imagecodecs

In [8]:
 #single image plot test
import numpy as np
import matplotlib.pyplot as plt
from skimage.measure import regionprops
from cellpose import models, plot, io, core
import pandas as pd
import seaborn as sns
import os
import imagecodecs
from pathlib import Path
from natsort import natsorted

io.logger_setup() # run this to get printing of progress

#Check if colab notebook instance has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime to one with GPU")

2026-03-23 19:41:36,668 [INFO] WRITING LOG OUTPUT TO /root/.cellpose/run.log
2026-03-23 19:41:36,669 [INFO] 
cellpose version: 	4.0.9 
platform:       	linux 
python version: 	3.12.12 
torch version:  	2.10.0+cu128
2026-03-23 19:41:36,671 [INFO] ** TORCH CUDA version installed and working. **


In [9]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
!ls /content/drive/MyDrive/DATA/

Mounted at /content/drive
images


In [10]:
# Loop through folders and find images
dir_path = "/content/drive/MyDrive/DATA/images/Psychatg02_07112025"
dir_obj = Path(dir_path)

if not dir_obj.exists():
    raise FileNotFoundError(f"Directory not found: {dir_path}")

# 2. List the files
image_ext = ".tif"
all_files = list(dir_obj.rglob("*" + image_ext))
files = natsorted([f for f in all_files if "_masks" not in f.name and "_flows" not in f.name])

# 3. Check if files were found
if len(files) == 0:
    raise FileNotFoundError(f"No {image_ext} files found in {dir_path}")
else:
    print(f"{len(files)} images found.")
    print(f"Success! Found {len(files)} total images across all subfolders.")
    # Show the first 3 paths to verify they are correct
    for f in files[:3]:
        print(f"Found: {f}")

90 images found.
Success! Found 90 total images across all subfolders.
Found: /content/drive/MyDrive/DATA/images/Psychatg02_07112025/Psychatg02_07112025/B+_w1/W001/P00001/HM_W001_P00001_CH2.tif
Found: /content/drive/MyDrive/DATA/images/Psychatg02_07112025/Psychatg02_07112025/B+_w1/W001/P00001/HM_W001_P00001_CH4.tif
Found: /content/drive/MyDrive/DATA/images/Psychatg02_07112025/Psychatg02_07112025/B+_w1/W001/P00001/HM_W001_P00001_Overlay.tif


In [11]:
# 1. Define your base path and conditions
study_map = {
    "Control": "B+_w1",
    "Condition": "S+B+_w1"
}

# 2. Initialize a dictionary to hold the separated file paths
separated_files = {key: [] for key in study_map.keys()}

# 3. Loop through the folders and sort files
# We use rglob to find all .tif files recursively
for file_path in dir_obj.rglob("*.tif"):
    # Skip masks or flows if they exist to avoid duplicates
    if "_masks" in file_path.name or "_flows" in file_path.name:
        continue

    # Check which condition folder the file belongs to
    for label, folder_keyword in study_map.items():
        if folder_keyword in str(file_path):
            separated_files[label].append(file_path)

# 4. Verify the results
for label, files in separated_files.items():
    print(f"{label}: found {len(files)} images.")
    if files:
        print(f"   Example: {files[0].name}")

Control: found 36 images.
   Example: HM_W001_P00002_CH2.tif
Condition: found 18 images.
   Example: HM_W001_P00001_Overlay.tif


In [12]:
# Updated function: Removed the 'rows = []' reset from inside
def run_cell_analysis(ch2_path, condition_label):
    """
    Performs Cellpose segmentation and feature extraction.
    Appends to the global 'rows' list without resetting it.
    """
    print(f"--- Analyzing: {os.path.basename(ch2_path)} ---")
    global rows
    global primary_id

    p = Path(ch2_path)
    ch4_path = p.with_name(p.name.replace("_CH2.tif", "_CH4.tif"))

    if not ch4_path.exists():
        print(f"Skipping: CH4 pair not found for {p.name}")
        return

    # Load Images
    ch2_img = io.imread(str(p))
    ch4_img = io.imread(str(ch4_path))

    # Run Cellpose
    masks, flows, styles = model.eval([ch2_img], diameter=30, channels=[0,0])
    masks = masks[0]

    # Measure Properties
    props = regionprops(masks, intensity_image=ch2_img)
    ch4_props = regionprops(masks, intensity_image=ch4_img)

    file_name = p.stem # Keeps the name without extension

    for p_gfp, p_s647 in zip(props, ch4_props):
        rows.append({
            "file": file_name,
            "condition": condition_label,
            "cell_id": p_gfp.label,
            "area_px": p_gfp.area,
            "mean_gfp": p_gfp.mean_intensity,
            "mean_s647": p_s647.mean_intensity
        })
    primary_id += 1

In [13]:
# --- Execute the function for both channels ---
# 1. Run analysis for CH2
# CH_basepath = "C:/Users/labadmin/Documents/BZ-X800/Chris/Psychatg02_07112025/"
# ch2_path = os.path.join(CH_basepath, filename + "_CH2.tif")
# CH2 = img
# CH4 = "/content/drive/MyDrive/cellpose_data/Psychatg02_07112025/S+B+_w1/W001/P00001/HM_W001_P00001_CH4.tif"

# run_cell_analysis(CH2)
# df_Splus = pd.DataFrame(rows)

# CH2 = img2
# CH4 = "/content/drive/MyDrive/cellpose_data/Psychatg02_07112025/B+_w1/W001/P00001/HM_W001_P00001_CH4.tif"
# run_cell_analysis(CH2)
# df_Sminus = pd.DataFrame(rows)
# df_all = pd.DataFrame(rows)

# print(f"df_Splus now has {len(df_Splus)} rows.")
# print(f"df_Sminus now has {len(df_Sminus)} rows.")

In [ ]:
# 1. Initialize the Cellpose model (This was missing!)
# 'cyto' or 'cyto2' are standard; use gpu=True for speed
model = models.CellposeModel(gpu=True)

# 2. Initialize data storage
rows = []
primary_id = 1

# 3. Loop through every condition in your study map
for condition, file_list in separated_files.items():
    print(f"\nProcessing group: {condition}")

    # Filter for CH2 files to use as the segmentation seed
    ch2_files = [f for f in file_list if "_CH2.tif" in f.name]

    for file_path in ch2_files:
        # The run_cell_analysis function handles finding the matching CH4
        run_cell_analysis(str(file_path), condition)

# 4. Create the final combined DataFrame
df_combined = pd.DataFrame(rows)

# 5. Clean up intensities (extracting scalar from RGB arrays if necessary)
def get_scalar(val):
    if isinstance(val, (list, np.ndarray)) and len(val) > 1:
        return val[1]
    return val

if not df_combined.empty:
    df_combined['mean_gfp'] = df_combined['mean_gfp'].apply(get_scalar)
    df_combined['mean_s647'] = df_combined['mean_s647'].apply(get_scalar)

    # Add a ratio column for easy analysis
    df_combined['ratio_647_gfp'] = df_combined['mean_s647'] / (df_combined['mean_gfp'] + 1e-9)

    print(f"\nDone! Total cells analyzed: {len(df_combined)}")
    display(df_combined.head())
else:
    print("No data collected. Check if CH2 and CH4 file pairs are named correctly.")

2026-03-23 19:43:20,084 [INFO] ** TORCH CUDA version installed and working. **
2026-03-23 19:43:20,087 [INFO] >>>> using GPU (CUDA)
2026-03-23 19:43:23,563 [INFO] Downloading: "https://huggingface.co/mouseland/cellpose-sam/resolve/main/cpsam" to /root/.cellpose/models/cpsam



100%|██████████| 1.15G/1.15G [00:05<00:00, 223MB/s]



Processing group: Control
--- Analyzing: HM_W001_P00002_CH2.tif ---
2026-03-23 19:43:30,459 [WARNING] channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
--- Analyzing: HM_W001_P00004_CH2.tif ---
2026-03-23 19:43:38,386 [WARNING] channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
--- Analyzing: HM_W001_P00005_CH2.tif ---
2026-03-23 19:43:46,091 [WARNING] channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
--- Analyzing: HM_W001_P00003_CH2.tif ---
2026-03-23 19:43:53,736 [WARNING] channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
--- Analyzing: HM_W001_P00001_CH2.tif ---
2026-03-23 19:44:01,178 [WARNING] channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
--- Analyzing: HM_W001_P00006_CH2.tif ---
2026-0

In [ ]:
df = pd.DataFrame(rows)
df

In [ ]:
df_Splus

In [ ]:
df_Sminus

In [ ]:
#concat the S+ and S-
df_Splus['Condition'] = 'S+B+'
df_Sminus['Condition'] = 'S-B+'

# Combine the two DataFrames
df_combined = pd.concat([df_Splus, df_Sminus], ignore_index=True)

# Extract the scalar mean intensity from the arrays before calculating the ratio
df_combined['mean_gfp_scalar'] = df_combined['mean_gfp'].apply(lambda x: x[1] if isinstance(x, np.ndarray) and len(x) > 1 else x)
df_combined['mean_s647_scalar'] = df_combined['mean_s647'].apply(lambda x: x[1] if isinstance(x, np.ndarray) and len(x) > 1 else x)

df_combined['ratio_s647_gfp'] = df_combined['mean_s647_scalar'] / (df_combined['mean_gfp_scalar'] + 1e-9)

In [ ]:
#violin plot
sns.violinplot(
    data=df_combined,
    x='Condition',
    y='ratio_s647_gfp',
    hue='Condition',  # ADDED: Assign x to hue as recommended
    legend=False,     # ADDED: Hide the redundant legend
    palette=['blue', 'gray'],
    order=['S-B+', 'S+B+']
)

In [ ]:
df_combined

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=df_combined,
    x='mean_gfp_scalar',
    y='mean_s647_scalar',
    hue='Condition',  # Use the new column to distinguish colors
    alpha=0.7         # Set transparency for better visibility of overlapping points
)

plt.xlabel("GFP Intensity")
plt.ylabel("SA_647 Intensity")
plt.title("Correlation of GFP vs. CH4 Intensity per Cell")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
fig = plt.figure(figsize=(12,5))
plot.show_segmentation(fig, ch1_img, masks, flows[0])
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.plot([1, 2, 3], [4, 5, 6])
plt.savefig("figure.svg")   # SVG vector
plt.savefig("figure.pdf")   # PDF vector
plt.savefig("figure.eps")   # EPS vector


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Drop non-numeric columns automatically
numeric_df = df.select_dtypes(include=['number'])

# Run PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(numeric_df)

# Plot
plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], s=40)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of Cell Features")
plt.tight_layout()
plt.show()
